# เตรียมข้อมูลพยากรณ์ตลาดและอัตราการเข้าพัก

ต่อจาก Notebook 02 ใช้ข้อมูลจำลองใน `data/processed/` เตรียมสำหรับ **SARIMA, SARIMAX และ Random Forest**
พร้อม Naïve และ Seasonal Naïve เป็นวิธีพื้นฐานสำหรับเปรียบเทียบ ยังไม่ฝึกหรือเลือกโมเดล

อ่านและ Run All ตามลำดับ 12 ขั้นตอน โค้ดที่เรียกซ้ำแยกไว้ใน
`src/kaverentai/features/market_occupancy_features.py` มีคำอธิบายภาษาไทย
ส่วน Notebook นี้แสดงตัวอย่างก่อน–หลังแต่ละขั้นตอน

ใช้ `weekly_market_cleaned.csv` เป็นตารางหลัก และ `projects_cleaned.csv` เพิ่มข้อมูลโครงการ
ไม่ใช้ units/leases สร้าง occupancy ใหม่ เนื่องจาก EDA พบการนับสัญญายังไม่ตรง snapshot ตลาด

1, 4, 12 สัปดาห์เป็น **ช่วงทดลองของโครงงาน** ไม่ใช่ช่วงที่งานวิจัยกำหนดว่าเหมาะที่สุด
ผลข้อมูลอยู่ใน `data/modeling/market_occupancy/` และสรุปใน `reports/market_occupancy/features/`

## 1. Import และตั้งค่า

หาตำแหน่งโปรเจกต์อัตโนมัติ เปิดได้ทั้งจาก root และ notebooks
ใช้ไฟล์ config เดียวกันกำหนด horizon และวันแบ่งข้อมูล

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import yaml
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "data" / "processed").is_dir():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("เปิด Notebook ภายในโฟลเดอร์โปรเจกต์")
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from kaverentai.features.market_occupancy_features import (
    TARGETS, LAGS, WINDOWS, prepare_weekly_panel, build_origin_features,
    build_forecast_pairs, label_time_splits, training_available_at_origin,
)

with (PROJECT_ROOT / "configs" / "model_config.yaml").open(encoding="utf-8") as file:
    config = yaml.safe_load(file)

HORIZONS = config["forecasting"]["horizons_weeks"]
TRAIN_END = config["time_split"]["train_end"]
VALIDATION_END = config["time_split"]["validation_end"]
TEST_END = config["forecasting"]["test_end"]
SEASONAL_PERIOD = config["forecasting"]["seasonal_period_weeks"]
MIN_SEASONAL_HISTORY = 2 * SEASONAL_PERIOD

MODEL_DATA_DIR = PROJECT_ROOT / "data" / "modeling" / "market_occupancy"
REPORT_DIR = PROJECT_ROOT / "reports" / "market_occupancy" / "features"
MODEL_DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Targets:", TARGETS)
print("Horizons:", HORIZONS)
print("Train / Validation / Test สิ้นสุด:", TRAIN_END, VALIDATION_END, TEST_END)

Targets: ['occupancy', 'n_searchers', 'n_listings_open', 'median_asking_rent']
Horizons: [1, 4, 12]
Train / Validation / Test สิ้นสุด: 2025-12-31 2026-03-31 2026-06-30


## 2. โหลดเฉพาะข้อมูลที่ใช้จริง

ขั้นเตรียมข้อมูลใช้ 2 ไฟล์ และไม่แก้ CSV ต้นฉบับ
วัน `date` ถือเป็นวันออกรายงานที่ตัวเลขสัปดาห์นั้นพร้อมใช้แล้ว
ถ้าใช้งานจริงมีความล่าช้าในการรายงาน ต้องเพิ่ม available_at และเลื่อน input ตามเวลาที่ได้รับจริง

In [2]:
DATA_DIR = PROJECT_ROOT / "data" / "processed"
weekly = pd.read_csv(DATA_DIR / "weekly_market_cleaned.csv")
projects = pd.read_csv(DATA_DIR / "projects_cleaned.csv")

display(weekly.head())
display(projects[["project_id", "project_name", "total_units"]].head())
print("weekly_market:", len(weekly), "แถว")
print("projects:", len(projects), "โครงการ")

,week,date,project_id,season,n_searchers,n_listings_open,n_units_leased,occupancy,median_asking_rent
0,0,2023-01-03,0,normal,21,101,488,0.828523,10010.0
1,0,2023-01-03,1,normal,39,166,907,0.845294,10480.0
2,0,2023-01-03,2,normal,36,180,893,0.832246,10965.0
3,0,2023-01-03,3,normal,55,285,1485,0.838983,11110.0
4,0,2023-01-03,4,normal,40,155,928,0.856879,11260.0


,project_id,project_name,total_units
0,0,Kave Condo,589
1,1,Kave Town Space,1073
2,2,Kave Town Shift,1073
3,3,Kave Town Island,1770
4,4,Kave Town Colony,1083


weekly_market: 2288 แถว
projects: 19 โครงการ


## 3. จัดตารางรายสัปดาห์แยกโครงการ

ฟังก์ชัน prepare_weekly_panel ทำ 3 อย่าง:
1. ตรวจ date และ key ซ้ำ
2. สร้างตารางวันที่ห่างกัน 7 วันภายในช่วงของแต่ละโครงการ หากสัปดาห์หายให้เป็นแถว NaN
3. เชื่อมชื่อโครงการและจำนวนห้องแบบ many-to-one

ไม่สร้างสัปดาห์ก่อนที่โครงการมีข้อมูล และไม่ใช้ shift ข้ามโครงการ

In [3]:
panel = prepare_weekly_panel(weekly, projects)

panel_overview = panel.groupby("project_id").agg(
    first_date=("date", "min"),
    last_date=("date", "max"),
    weekly_rows=("date", "size"),
    observed_weeks=("is_observed_week", "sum"),
).reset_index()
panel_overview["missing_weeks"] = panel_overview["weekly_rows"] - panel_overview["observed_weeks"]
display(panel_overview)
display(panel[["project_id", "date", "project_name"] + TARGETS].head())
assert not panel.duplicated(["project_id", "date"]).any()

,project_id,first_date,last_date,weekly_rows,observed_weeks,missing_weeks
0,0,2023-01-03,2026-06-23,182,182,0
1,1,2023-01-03,2026-06-23,182,182,0
2,2,2023-01-03,2026-06-23,182,182,0
3,3,2023-01-03,2026-06-23,182,182,0
4,4,2023-01-03,2026-06-23,182,182,0
5,5,2023-01-03,2026-06-23,182,182,0
6,6,2023-01-03,2026-06-23,182,182,0
7,7,2024-12-31,2026-06-23,78,78,0
8,8,2025-12-30,2026-06-23,26,26,0
9,9,2024-01-02,2026-06-23,130,130,0


,project_id,date,project_name,occupancy,n_searchers,n_listings_open,median_asking_rent
0,0,2023-01-03,Kave Condo,0.828523,21,101,10010.0
1,0,2023-01-10,Kave Condo,0.828523,22,101,10050.0
2,0,2023-01-17,Kave Condo,0.831919,17,99,10060.0
3,0,2023-01-24,Kave Condo,0.840407,20,94,10135.0
4,0,2023-01-31,Kave Condo,0.845501,20,91,9930.0


## 4. จัดการราคาเสนอเช่าที่ว่างอย่างแยกหน้าที่

สร้าง `asking_rent_input` ด้วย forward fill ภายในโครงการ ใช้เฉพาะราคาที่เคยรู้แล้ว
เก็บ `asking_rent_missing_at_origin` เพื่อบอกว่า ณ สัปดาห์นี้ไม่มีราคาจริง
คอลัมน์ `median_asking_rent` เดิมยังคงว่างเพื่อไม่แต่งคำตอบสำหรับประเมิน

`build_origin_features` สร้างตัวแปรย้อนหลังในขั้น 5 พร้อมกัน ส่วนนี้ตรวจผลการเติมราคาให้เห็นก่อน

In [4]:
origin_features, numeric_features, categorical_features = build_origin_features(panel)
missing_price_examples = origin_features.loc[
    origin_features["median_asking_rent"].isna(),
    ["project_id", "date", "n_listings_open", "median_asking_rent", "asking_rent_input", "asking_rent_missing_at_origin"],
]
display(missing_price_examples)

# ตรวจว่าราคา target เดิมไม่ได้ถูกเติม
assert origin_features["median_asking_rent"].isna().sum() == panel["median_asking_rent"].isna().sum()

,project_id,date,n_listings_open,median_asking_rent,asking_rent_input,asking_rent_missing_at_origin
677,3,2025-07-08,0,NaN,11910.0,1
1308,7,2025-08-26,0,NaN,13285.0,1
1309,7,2025-09-02,0,NaN,13285.0,1


## 5. ดูตัวแปรย้อนหลังสำหรับ Random Forest

- `_current`: ค่าที่รู้ ณ origin
- `_lag_1 / 4 / 12`: ค่าเมื่อ 1 / 4 / 12 สัปดาห์ก่อน
- `_rolling_mean_4 / 12`: ค่าเฉลี่ย 4 / 12 สัปดาห์ก่อน origin **ไม่รวมสัปดาห์ปัจจุบัน**

ตัวอย่าง ถ้า origin คือ 2 เมษายน ค่า lag_1 มาจาก 26 มีนาคม
rolling_mean_4 มาจาก 4 สัปดาห์ก่อน 2 เมษายน ไม่อ่านข้อมูลหลัง origin
จำนวน lag/window เป็นตัวเลือกเริ่มต้นที่ต้องตรวจด้วย Validation ภายหลัง

ถ้าราคา input ยังว่างก่อนมีราคาครั้งแรก จะคง NaN ให้ขั้นฝึก fit imputer บนชุดฝึกของรอบนั้น
ไม่คำนวณ median หรือ scaling จากข้อมูลทั้งชุดใน Notebook นี้

In [5]:
example_columns = [
    "project_id", "date", "occupancy_current", "occupancy_lag_1",
    "occupancy_lag_4", "occupancy_lag_12", "occupancy_rolling_mean_4",
    "occupancy_rolling_mean_12", "history_complete_12",
]
example_project = origin_features["project_id"].min()
display(origin_features.loc[origin_features["project_id"].eq(example_project), example_columns].head(16))
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

,project_id,date,occupancy_current,occupancy_lag_1,occupancy_lag_4,occupancy_lag_12,occupancy_rolling_mean_4,occupancy_rolling_mean_12,history_complete_12
0,0,2023-01-03,0.828523,NaN,NaN,NaN,NaN,NaN,False
1,0,2023-01-10,0.828523,0.828523,NaN,NaN,NaN,NaN,False
2,0,2023-01-17,0.831919,0.828523,NaN,NaN,NaN,NaN,False
3,0,2023-01-24,0.840407,0.831919,NaN,NaN,NaN,NaN,False
4,0,2023-01-31,0.845501,0.840407,0.828523,NaN,0.832343,NaN,False
5,0,2023-02-07,0.843803,0.845501,0.828523,NaN,0.836587,NaN,False
6,0,2023-02-14,0.842105,0.843803,0.831919,NaN,0.840407,NaN,False
7,0,2023-02-21,0.833616,0.842105,0.840407,NaN,0.842954,NaN,False
8,0,2023-02-28,0.826825,0.833616,0.845501,NaN,0.841256,NaN,False
9,0,2023-03-07,0.826825,0.826825,0.843803,NaN,0.836587,NaN,False


Numeric features: ['total_units', 'distance_to_campus_m', 'facility_count', 'asking_rent_missing_at_origin', 'occupancy_current', 'occupancy_lag_1', 'occupancy_lag_4', 'occupancy_lag_12', 'occupancy_rolling_mean_4', 'occupancy_rolling_mean_12', 'n_searchers_current', 'n_searchers_lag_1', 'n_searchers_lag_4', 'n_searchers_lag_12', 'n_searchers_rolling_mean_4', 'n_searchers_rolling_mean_12', 'n_listings_open_current', 'n_listings_open_lag_1', 'n_listings_open_lag_4', 'n_listings_open_lag_12', 'n_listings_open_rolling_mean_4', 'n_listings_open_rolling_mean_12', 'median_asking_rent_current', 'median_asking_rent_lag_1', 'median_asking_rent_lag_4', 'median_asking_rent_lag_12', 'median_asking_rent_rolling_mean_4', 'median_asking_rent_rolling_mean_12', 'observed_weeks_at_origin', 'origin_month', 'origin_year']
Categorical features: ['project_key', 'university', 'season']


## 6. สร้างเป้าหมายด้วยวันที่จริง

กำหนด `target_date = forecast_origin + horizon` แล้ว merge คำตอบจากโครงการและวันที่นั้น
เก็บคำตอบ 4 ตัวเป็น `y_...` และไม่เติมค่าว่าง target
`target_month_sin/cos` เป็นเดือนของวันเป้าหมายที่ทราบล่วงหน้าจากปฏิทิน
ไม่ใช้ season ในอนาคตจาก CSV เป็น input

In [6]:
pairs_by_horizon = {}
for horizon in HORIZONS:
    pairs = build_forecast_pairs(origin_features, panel, horizon)
    pairs = label_time_splits(pairs, TRAIN_END, VALIDATION_END, TEST_END)
    pairs_by_horizon[horizon] = pairs

preview = pairs_by_horizon[4][[
    "project_id", "forecast_origin", "target_date", "horizon_weeks",
    "occupancy_current", "y_occupancy", "split",
]]
display(preview.head(8))

for horizon, pairs in pairs_by_horizon.items():
    assert (pairs["target_date"] - pairs["forecast_origin"]).eq(pd.Timedelta(weeks=horizon)).all()

,project_id,forecast_origin,target_date,horizon_weeks,occupancy_current,y_occupancy,split
0,0,2023-01-03,2023-01-31,4,0.828523,0.845501,Train
1,0,2023-01-10,2023-02-07,4,0.828523,0.843803,Train
2,0,2023-01-17,2023-02-14,4,0.831919,0.842105,Train
3,0,2023-01-24,2023-02-21,4,0.840407,0.833616,Train
4,0,2023-01-31,2023-02-28,4,0.845501,0.826825,Train
5,0,2023-02-07,2023-03-07,4,0.843803,0.826825,Train
6,0,2023-02-14,2023-03-14,4,0.842105,0.826825,Train
7,0,2023-02-21,2023-03-21,4,0.833616,0.823430,Train


## 7. เตรียมตาราง Random Forest แยก target และ horizon

ใช้เฉพาะ origin ที่เป็นสัปดาห์จริง มี occupancy ย้อนหลังครบ 12 สัปดาห์ และมีคำตอบของ target นั้น
ถ้าราคา target ว่าง ตัดเฉพาะงานพยากรณ์ราคา ไม่ลบแถวนั้นจากงาน occupancy/ผู้ค้นหา/ประกาศ
ท้ายข้อมูลที่ยังไม่มีคำตอบในอนาคตยังไม่ใช่ตัวอย่างฝึก และถูกนับในรายงานการคัดแถว

เลือก input ด้วยรายการชื่อที่กำหนดเท่านั้น ห้ามเลือกทุกคอลัมน์อัตโนมัติ เพราะตารางคู่มีคำตอบอนาคตอยู่ด้วย

In [7]:
calendar_features = ["target_month_sin", "target_month_cos"]
rf_features = numeric_features + categorical_features + calendar_features
date_columns = ["project_id", "forecast_origin", "target_date", "horizon_weeks", "split"]
rf_datasets = {}
row_selection = []

for horizon, pairs in pairs_by_horizon.items():
    valid_origin = pairs["is_observed_week"] & pairs["history_complete_12"]
    inside_period = pairs["split"].ne("outside_period")
    for target in TARGETS:
        has_answer = pairs[f"y_{target}"].notna()
        keep = valid_origin & inside_period & has_answer
        dataset = pairs.loc[keep, date_columns + rf_features + [f"y_{target}"]].copy()
        dataset = dataset.rename(columns={f"y_{target}": "y"})
        rf_datasets[(target, horizon)] = dataset
        row_selection.append({
            "target": target, "horizon_weeks": horizon,
            "total_origins": len(pairs),
            "invalid_or_short_history": int((~valid_origin).sum()),
            "outside_period_after_history": int((valid_origin & ~inside_period).sum()),
            "missing_target_inside_period": int((valid_origin & inside_period & ~has_answer).sum()),
            "kept_rows": len(dataset),
        })

row_selection = pd.DataFrame(row_selection)
display(row_selection)
display(rf_datasets[("occupancy", 4)].head())
assert not any(column.startswith("y_") for column in rf_features)

,target,horizon_weeks,total_origins,invalid_or_short_history,outside_period_after_history,missing_target_inside_period,kept_rows
0,occupancy,1,2288,216,0,18,2054
1,n_searchers,1,2288,216,0,18,2054
2,n_listings_open,1,2288,216,0,18,2054
3,median_asking_rent,1,2288,216,0,21,2051
4,occupancy,4,2288,216,54,18,2000
5,n_searchers,4,2288,216,54,18,2000
6,n_listings_open,4,2288,216,54,18,2000
7,median_asking_rent,4,2288,216,54,21,1997
8,occupancy,12,2288,216,198,18,1856
9,n_searchers,12,2288,216,198,18,1856


,project_id,forecast_origin,target_date,horizon_weeks,split,total_units,distance_to_campus_m,facility_count,asking_rent_missing_at_origin,occupancy_current,occupancy_lag_1,occupancy_lag_4,occupancy_lag_12,occupancy_rolling_mean_4,occupancy_rolling_mean_12,n_searchers_current,n_searchers_lag_1,n_searchers_lag_4,n_searchers_lag_12,n_searchers_rolling_mean_4,n_searchers_rolling_mean_12,n_listings_open_current,n_listings_open_lag_1,n_listings_open_lag_4,n_listings_open_lag_12,n_listings_open_rolling_mean_4,n_listings_open_rolling_mean_12,median_asking_rent_current,median_asking_rent_lag_1,median_asking_rent_lag_4,median_asking_rent_lag_12,median_asking_rent_rolling_mean_4,median_asking_rent_rolling_mean_12,observed_weeks_at_origin,origin_month,origin_year,project_key,university,season,target_month_sin,target_month_cos,y
12,0,2023-03-28,2023-04-25,4,Train,589,100,20,0,0.818336,0.823430,0.826825,0.828523,0.825976,0.833192,10,7.0,7.0,21.0,8.50,13.250000,107,104.0,102.0,101.0,102.50,98.250000,9430.0,9550.0,9480.0,10010.0,9546.25,9820.416667,13,3,2023,0,ม.กรุงเทพ,off,1.000000,6.123234e-17,0.818336
13,0,2023-04-04,2023-05-02,4,Train,589,100,20,0,0.826825,0.818336,0.826825,0.828523,0.823854,0.832343,13,10.0,8.0,22.0,9.25,12.333333,102,107.0,102.0,101.0,103.75,98.750000,9695.0,9430.0,9550.0,10050.0,9533.75,9772.083333,14,4,2023,0,ม.กรุงเทพ,off,0.866025,-5.000000e-01,0.814941
14,0,2023-04-11,2023-05-09,4,Train,589,100,20,0,0.826825,0.826825,0.826825,0.831919,0.823854,0.832201,12,13.0,12.0,17.0,10.50,11.583333,102,102.0,102.0,99.0,103.75,98.833333,9405.0,9695.0,9605.0,10060.0,9570.00,9742.500000,15,4,2023,0,ม.กรุงเทพ,off,0.866025,-5.000000e-01,0.847199
15,0,2023-04-18,2023-05-16,4,Train,589,100,20,0,0.816638,0.826825,0.823430,0.840407,0.823854,0.831777,8,12.0,7.0,20.0,10.50,11.166667,108,102.0,104.0,94.0,103.75,99.083333,9315.0,9405.0,9550.0,10135.0,9520.00,9687.916667,16,4,2023,0,ม.กรุงเทพ,off,0.866025,-5.000000e-01,0.877759
16,0,2023-04-25,2023-05-23,4,Train,589,100,20,0,0.818336,0.816638,0.818336,0.845501,0.822156,0.829796,11,8.0,10.0,20.0,10.75,10.166667,107,108.0,107.0,91.0,104.75,100.250000,9400.0,9315.0,9430.0,9930.0,9461.25,9619.583333,17,4,2023,0,ม.กรุงเทพ,off,0.866025,-5.000000e-01,0.899830


## 8. เตรียมอนุกรมเวลาสำหรับ SARIMA และ Seasonal Naïve

SARIMA ใช้ y รายสัปดาห์แยกโครงการ ไม่ใช้ตาราง lag ของ Random Forest
คงราคา y ที่ว่างไว้ state-space fitting ต้องจัดการ missing endog ในขั้นฝึก โดยไม่เติมคำตอบสำหรับประเมิน

ทดลองฤดูกาล 52 สัปดาห์เป็นประมาณหนึ่งปี ไม่ใช่ horizon และไม่ใช่ข้อกำหนดจากงานวิจัย
สำหรับการเปรียบเทียบหลัก กำหนดประวัติอย่างน้อย 2 รอบ หรือ 104 จุดที่มี y เป็นเกณฑ์เริ่มต้นแบบระมัดระวัง
เกณฑ์นี้เป็นการออกแบบโครงงาน ไม่ใช่หลักประกันว่าฝึกได้หรือแม่น ต้องตรวจการลู่เข้าและ Validation อีกครั้ง
โครงการที่มีข้อมูลน้อยกว่าจะรายงานแยกในกลุ่มประวัติสั้น

In [8]:
sarima_series = panel[["project_id", "date", "is_observed_week"] + TARGETS].copy()
sarima_series["split"] = "Test"
sarima_series.loc[sarima_series["date"].le(pd.Timestamp(VALIDATION_END)), "split"] = "Validation"
sarima_series.loc[sarima_series["date"].le(pd.Timestamp(TRAIN_END)), "split"] = "Train"

display(sarima_series.head())
print("Seasonal period ทดลอง:", SEASONAL_PERIOD, "สัปดาห์")
print("ประวัติขั้นต่ำสำหรับการเปรียบเทียบหลัก:", MIN_SEASONAL_HISTORY, "จุดที่มี y")

# เมื่อเริ่มพยากรณ์จริงต้องตัดอนุกรมเวลาที่ date <= origin เสมอ
example_origin = pd.Timestamp("2025-10-14")
example_series = sarima_series.loc[
    sarima_series["project_id"].eq(example_project) & sarima_series["date"].le(example_origin),
    ["date", "occupancy"],
].set_index("date").asfreq("7D")
display(example_series.tail())
assert example_series.index.max() <= example_origin

,project_id,date,is_observed_week,occupancy,n_searchers,n_listings_open,median_asking_rent,split
0,0,2023-01-03,True,0.828523,21,101,10010.0,Train
1,0,2023-01-10,True,0.828523,22,101,10050.0,Train
2,0,2023-01-17,True,0.831919,17,99,10060.0,Train
3,0,2023-01-24,True,0.840407,20,94,10135.0,Train
4,0,2023-01-31,True,0.845501,20,91,9930.0,Train


Seasonal period ทดลอง: 52 สัปดาห์
ประวัติขั้นต่ำสำหรับการเปรียบเทียบหลัก: 104 จุดที่มี y


,occupancy
date,
2025-09-16,0.933786
2025-09-23,0.937182
2025-09-30,0.926995
2025-10-07,0.932088
2025-10-14,0.933786


## 9. เตรียมตัวแปรภายนอกสำหรับ SARIMAX แบบใช้ได้ ณ origin

สร้างแบบจำลองแยกแต่ละ horizon: สำหรับ y ณ วัน t ใช้ตลาดที่ทราบ ณ t−h
occupancy ใช้จำนวนผู้ค้นหาและราคา ส่วน target อื่นใช้ occupancy และราคา/ผู้ค้นหาที่ไม่ใช่ตัว target เอง
ไม่ใส่ n_listings_open เป็นตัวประกอบของ occupancy เพราะในข้อมูลนี้มีสูตรผูกกันโดยตรง

เพิ่มเดือนเป้าหมายจากปฏิทิน ตอนพยากรณ์ h ก้าวต้องส่ง exog ครบทุกก้าว ไม่ใช่แค่วันสุดท้าย
ตัวอย่างนี้เลื่อนข้อมูลประกอบย้อนหลัง h สัปดาห์ จึงรู้ค่าตลอดเส้นทางโดยไม่ต้องหยิบค่าตลาดในอนาคตจริง
ตารางฝึกมี y แต่ **ตาราง exog ที่ส่งตอน predict ต้องเลือกเฉพาะ x_ เท่านั้น**

In [9]:
exogenous_inputs = {
    "occupancy": ["n_searchers_current", "median_asking_rent_current"],
    "n_searchers": ["occupancy_current", "median_asking_rent_current"],
    "n_listings_open": ["occupancy_current", "n_searchers_current"],
    "median_asking_rent": ["occupancy_current", "n_searchers_current"],
}
sarimax_datasets = {}
sarimax_feature_names = {}

for horizon, pairs in pairs_by_horizon.items():
    for target in TARGETS:
        source_columns = exogenous_inputs[target] + calendar_features
        column_names = {name: f"x_{name}" for name in source_columns}
        columns = date_columns + ["history_complete_12", "is_observed_week"] + source_columns + [f"y_{target}"]
        dataset = pairs[columns].rename(columns={**column_names, f"y_{target}": "y"}).copy()
        x_columns = list(column_names.values())
        dataset["exog_complete"] = dataset[x_columns].notna().all(axis=1)
        sarimax_datasets[(target, horizon)] = dataset
        sarimax_feature_names[target] = x_columns

display(sarimax_datasets[("occupancy", 4)].head(8))

# ตัวอย่าง exog ที่ใช้พยากรณ์จาก origin ไป 4 ก้าว โดยไม่มี y ปน
origin = pd.Timestamp("2025-10-14")
expected_dates = pd.date_range(origin + pd.Timedelta(weeks=1), periods=4, freq="7D")
exog_source = sarimax_datasets[("occupancy", 4)]
path_rows = exog_source.loc[
    exog_source["project_id"].eq(example_project) & exog_source["target_date"].isin(expected_dates)
].sort_values("target_date")
display(path_rows[["forecast_origin", "target_date"] + sarimax_feature_names["occupancy"]])
exog_for_predict = path_rows.set_index("target_date")[sarimax_feature_names["occupancy"]]
assert len(exog_for_predict) == 4
assert path_rows["forecast_origin"].le(origin).all()
assert "y" not in exog_for_predict.columns

,project_id,forecast_origin,target_date,horizon_weeks,split,history_complete_12,is_observed_week,x_n_searchers_current,x_median_asking_rent_current,x_target_month_sin,x_target_month_cos,y,exog_complete
0,0,2023-01-03,2023-01-31,4,Train,False,True,21,10010.0,0.000000,1.000000,0.845501,True
1,0,2023-01-10,2023-02-07,4,Train,False,True,22,10050.0,0.500000,0.866025,0.843803,True
2,0,2023-01-17,2023-02-14,4,Train,False,True,17,10060.0,0.500000,0.866025,0.842105,True
3,0,2023-01-24,2023-02-21,4,Train,False,True,20,10135.0,0.500000,0.866025,0.833616,True
4,0,2023-01-31,2023-02-28,4,Train,False,True,20,9930.0,0.500000,0.866025,0.826825,True
5,0,2023-02-07,2023-03-07,4,Train,False,True,10,9985.0,0.866025,0.500000,0.826825,True
6,0,2023-02-14,2023-03-14,4,Train,False,True,9,9770.0,0.866025,0.500000,0.826825,True
7,0,2023-02-21,2023-03-21,4,Train,False,True,6,9720.0,0.866025,0.500000,0.823430,True


,forecast_origin,target_date,x_n_searchers_current,x_median_asking_rent_current,x_target_month_sin,x_target_month_cos
142,2025-09-23,2025-10-21,25,10020.0,-1.000000,-1.836970e-16
143,2025-09-30,2025-10-28,32,10120.0,-1.000000,-1.836970e-16
144,2025-10-07,2025-11-04,39,10385.0,-0.866025,5.000000e-01
145,2025-10-14,2025-11-11,34,10060.0,-0.866025,5.000000e-01


## 10. ตรวจการแบ่งเวลาและ label leakage

Train ตาม target_date อย่างเดียว **ยังไม่เพียงพอ** สำหรับ backtesting หลายก้าว
ตัวอย่าง target แรกของ Validation ที่ horizon 12 สัปดาห์ มี origin ก่อนสิ้นปี 2025
หาก fit Train ทั้งหมดถึงสิ้นปีจะได้เห็นคำตอบที่ยังไม่เกิด ณ origin นั้น

`training_available_at_origin` จึงเลือก Train ที่ target_date <= origin ของแต่ละรอบอีกครั้ง
หลักเดียวกันใช้กับ SARIMA/SARIMAX และ imputer/encoder/scaler ต้อง fit เฉพาะชุดที่ทราบในรอบนั้น
กรอบเริ่มต้นนี้ไม่ใช้คำตอบ Validation ฝึกเพิ่มระหว่างเลือกโมเดล

In [10]:
split_rows = []
for (target, horizon), dataset in rf_datasets.items():
    for split, frame in dataset.groupby("split"):
        split_rows.append({
            "target": target, "horizon_weeks": horizon, "split": split,
            "rows": len(frame), "projects": frame["project_id"].nunique(),
            "first_target_date": frame["target_date"].min(),
            "last_target_date": frame["target_date"].max(),
        })
split_summary = pd.DataFrame(split_rows)
display(split_summary)

occupancy_12 = rf_datasets[("occupancy", 12)]
first_validation_origin = occupancy_12.loc[occupancy_12["split"].eq("Validation"), "forecast_origin"].min()
available_train = training_available_at_origin(occupancy_12, first_validation_origin)
future_labels_in_train = occupancy_12.loc[
    occupancy_12["split"].eq("Train") & occupancy_12["target_date"].gt(first_validation_origin)
]
print("origin แรกของ Validation ที่ h=12:", first_validation_origin.date())
print("Train ทั้งหมด:", occupancy_12["split"].eq("Train").sum(), "แถว")
print("Train ที่ใช้ได้ ณ origin นี้:", len(available_train), "แถว")
print("Train ที่ต้องกันออกเพราะยังไม่ทราบคำตอบ:", len(future_labels_in_train), "แถว")
assert available_train["target_date"].le(first_validation_origin).all()

,target,horizon_weeks,split,rows,projects,first_target_date,last_target_date
0,occupancy,1,Test,216,18,2026-04-07,2026-06-23
1,occupancy,1,Train,1640,15,2023-04-04,2025-12-30
2,occupancy,1,Validation,198,18,2026-01-06,2026-03-31
3,n_searchers,1,Test,216,18,2026-04-07,2026-06-23
4,n_searchers,1,Train,1640,15,2023-04-04,2025-12-30
5,n_searchers,1,Validation,198,18,2026-01-06,2026-03-31
6,n_listings_open,1,Test,216,18,2026-04-07,2026-06-23
7,n_listings_open,1,Train,1640,15,2023-04-04,2025-12-30
8,n_listings_open,1,Validation,198,18,2026-01-06,2026-03-31
9,median_asking_rent,1,Test,216,18,2026-04-07,2026-06-23


origin แรกของ Validation ที่ h=12: 2025-10-14
Train ทั้งหมด: 1475 แถว
Train ที่ใช้ได้ ณ origin นี้: 1310 แถว
Train ที่ต้องกันออกเพราะยังไม่ทราบคำตอบ: 165 แถว


## 11. ระบุโครงการประวัติสั้นและชุดเปรียบเทียบที่ยุติธรรม

Random Forest รวมข้อมูลหลายโครงการฝึกได้ แต่โครงการใหม่ยังต้องมี input ย้อนหลังครบก่อน
encoder ต้องรองรับ project_key ที่ไม่เคยพบในชุดฝึก เช่น OneHotEncoder(handle_unknown="ignore")
การใช้ pooled model ยังไม่รับประกันความแม่นยำของโครงการใหม่

สำหรับเปรียบเทียบ 5 วิธีหลัก ให้ใช้ project/origin/target/horizon เดียวกันที่มีประวัติ y อย่างน้อย 104 จุด
โครงการที่ยังไม่ผ่านเกณฑ์ให้แสดงผล Random Forest/Naïve แยก ไม่ใช้คะแนนชุดคนละแถวมาตัดสินว่าดีกว่า SARIMA
flag ด้านล่างเป็นความพร้อมเบื้องต้น ผล fit/lags/exog อาจทำให้ต้องกรองชุดร่วมเพิ่มในขั้นประเมิน

In [11]:
history = panel.loc[panel["date"].le(pd.Timestamp(TRAIN_END))].groupby("project_id").agg(
    train_weeks=("is_observed_week", "sum"),
    occupancy_points=("occupancy", "count"),
).reset_index()
project_readiness = projects[["project_id", "project_name"]].merge(history, on="project_id", how="left")
project_readiness[["train_weeks", "occupancy_points"]] = project_readiness[["train_weeks", "occupancy_points"]].fillna(0).astype(int)
project_readiness["initial_status"] = "มีประวัติสำหรับทดลองฤดูกาล 52 สัปดาห์"
project_readiness.loc[project_readiness["train_weeks"].lt(MIN_SEASONAL_HISTORY), "initial_status"] = "ประวัติสั้น: แยกผลจากชุดเปรียบเทียบหลัก"
project_readiness.loc[project_readiness["train_weeks"].le(12), "initial_status"] = "ยังไม่มี Train หลังสร้างประวัติย้อนหลัง 12 สัปดาห์"
project_readiness.loc[project_readiness["train_weeks"].eq(0), "initial_status"] = "ไม่มีข้อมูล Train"
display(project_readiness)

# นับจำนวน y ที่รู้จริงถึง origin แยก target รวมค่าว่างของราคาอย่างตรงไปตรงมา
history_counts = panel[["project_id", "date"]].copy()
for target in TARGETS:
    history_counts[f"observed_{target}"] = panel[target].notna().groupby(panel["project_id"]).cumsum()

comparison_rows = []
for (target, horizon), dataset in rf_datasets.items():
    count_column = f"observed_{target}"
    counts = history_counts[["project_id", "date", count_column]].rename(columns={"date": "forecast_origin"})
    dataset = dataset.merge(counts, on=["project_id", "forecast_origin"], validate="many_to_one")
    dataset["seasonal_trial_ready"] = dataset[count_column].ge(MIN_SEASONAL_HISTORY)
    rf_datasets[(target, horizon)] = dataset
    for split, frame in dataset.groupby("split"):
        comparison_rows.append({
            "target": target, "horizon_weeks": horizon, "split": split,
            "all_rf_ready_rows": len(frame),
            "seasonal_trial_ready_rows": int(frame["seasonal_trial_ready"].sum()),
            "short_history_rows": int((~frame["seasonal_trial_ready"]).sum()),
        })

comparison_readiness = pd.DataFrame(comparison_rows)
display(comparison_readiness)

,project_id,project_name,train_weeks,occupancy_points,initial_status
0,0,Kave Condo,157,157,มีประวัติสำหรับทดลองฤดูกาล 52 สัปดาห์
1,1,Kave Town Space,157,157,มีประวัติสำหรับทดลองฤดูกาล 52 สัปดาห์
2,2,Kave Town Shift,157,157,มีประวัติสำหรับทดลองฤดูกาล 52 สัปดาห์
3,3,Kave Town Island,157,157,มีประวัติสำหรับทดลองฤดูกาล 52 สัปดาห์
4,4,Kave Town Colony,157,157,มีประวัติสำหรับทดลองฤดูกาล 52 สัปดาห์
5,5,Kave TU,157,157,มีประวัติสำหรับทดลองฤดูกาล 52 สัปดาห์
6,6,Kave Ava,157,157,มีประวัติสำหรับทดลองฤดูกาล 52 สัปดาห์
7,7,Kave Wonderland,53,53,ประวัติสั้น: แยกผลจากชุดเปรียบเทียบหลัก
8,8,Kave Carnival,1,1,ยังไม่มี Train หลังสร้างประวัติย้อนหลัง 12 สัป...
9,9,Kave Embryo,105,105,มีประวัติสำหรับทดลองฤดูกาล 52 สัปดาห์


,target,horizon_weeks,split,all_rf_ready_rows,seasonal_trial_ready_rows,short_history_rows
0,occupancy,1,Test,216,144,72
1,occupancy,1,Train,1640,428,1212
2,occupancy,1,Validation,198,156,42
3,n_searchers,1,Test,216,144,72
4,n_searchers,1,Train,1640,428,1212
5,n_searchers,1,Validation,198,156,42
6,n_listings_open,1,Test,216,144,72
7,n_listings_open,1,Train,1640,428,1212
8,n_listings_open,1,Validation,198,156,42
9,median_asking_rent,1,Test,216,144,72


## 12. บันทึกข้อมูล รายการ features และผลตรวจ

- `rf_<target>_h<h>.csv`: ตาราง Random Forest พร้อม split และ y
- `sarimax_<target>_h<h>.csv`: ตาราง endog/exog ที่เลื่อนข้อมูลตาม horizon
- `sarima_weekly_series.csv`: อนุกรมเวลาเดิม ไม่เติม target
- `forecast_feature_metadata.json`: รายการ input และนโยบายเตรียมข้อมูล

ไฟล์เหล่านี้เป็นข้อมูลที่สร้างจากโค้ด ไม่ใช่โมเดลที่ฝึกแล้ว
ค่าคำตอบใน Validation/Test อยู่ในไฟล์เพื่อประเมินเท่านั้น ต้องใช้ตัวกรองเวลาในขั้น 10 ก่อน fit ทุกครั้ง

In [12]:
for (target, horizon), dataset in rf_datasets.items():
    dataset.to_csv(MODEL_DATA_DIR / f"rf_{target}_h{horizon}.csv", index=False, encoding="utf-8-sig")
for (target, horizon), dataset in sarimax_datasets.items():
    dataset.to_csv(MODEL_DATA_DIR / f"sarimax_{target}_h{horizon}.csv", index=False, encoding="utf-8-sig")
sarima_series.to_csv(MODEL_DATA_DIR / "sarima_weekly_series.csv", index=False, encoding="utf-8-sig")

metadata = {
    "status": "features_prepared_models_not_trained",
    "targets": TARGETS,
    "horizons_weeks": HORIZONS,
    "horizon_reason": "project experimental settings, not prescribed by cited studies",
    "models": ["sarima", "sarimax", "random_forest_regressor"],
    "baselines": ["naive", "seasonal_naive"],
    "train_end": TRAIN_END, "validation_end": VALIDATION_END, "test_end": TEST_END,
    "lags_weeks": LAGS, "rolling_windows_weeks": WINDOWS,
    "rolling_excludes_origin_week": True,
    "rf_numeric_features": numeric_features + calendar_features,
    "rf_categorical_features": categorical_features,
    "sarimax_features_by_target": sarimax_feature_names,
    "sarimax_alignment": "For each horizon h, exog at target t uses observations at t-h; supply all h future exog rows using only observations <= origin",
    "seasonal_period_weeks": SEASONAL_PERIOD,
    "min_seasonal_trial_points": MIN_SEASONAL_HISTORY,
    "price_input_policy": "within-project past-only forward fill; remaining input NaN imputed only inside each fitting fold",
    "price_target_policy": "never impute y; evaluate price only when true price exists",
    "training_rule": "split=Train AND target_date <= current forecast_origin; endog series date <= current origin",
    "availability_assumption": "weekly snapshot is known on date; real reporting delay must be modeled before production",
    "no_future_season_feature": True,
}
(MODEL_DATA_DIR / "forecast_feature_metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")

report_tables = {"panel_overview": panel_overview, "row_selection": row_selection,
                 "split_summary": split_summary, "project_readiness": project_readiness,
                 "comparison_readiness": comparison_readiness}
for name, table in report_tables.items():
    table.to_csv(REPORT_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")

summary = f"""# ผลเตรียมข้อมูลพยากรณ์ตลาดและอัตราการเข้าพัก

ใช้ weekly_market_cleaned.csv และ projects_cleaned.csv ซึ่งเป็นข้อมูลจำลอง

- ตารางรายสัปดาห์ {len(panel):,} แถว {panel['project_id'].nunique()} โครงการ
- ตาราง Random Forest {len(rf_datasets)} ชุด และ SARIMAX {len(sarimax_datasets)} ชุด สำหรับ {len(TARGETS)} targets × {len(HORIZONS)} horizons
- Numeric features {len(numeric_features + calendar_features)} ตัว และ categorical features {len(categorical_features)} ตัว
- ราคา target ว่าง {panel['median_asking_rent'].isna().sum()} แถว ไม่ถูกเติมเป็นคำตอบ
- origin แรกของ Validation ที่ h=12 คือ {first_validation_origin:%Y-%m-%d}; มี Train {len(future_labels_in_train):,} แถวที่ยังไม่ทราบคำตอบและต้องกันออกจากรอบนี้
- ทดลองฤดูกาล {SEASONAL_PERIOD} สัปดาห์ และเกณฑ์ชุดเปรียบเทียบหลัก {MIN_SEASONAL_HISTORY} จุดที่มี y; เป็นค่าตั้งต้นของโครงงานที่ยังต้องประเมิน
- โมเดลที่ตกลงทดลอง: SARIMA, SARIMAX, Random Forest; baseline: Naïve, Seasonal Naïve
- ยังไม่ได้ fit imputer, encoder, scaler หรือโมเดล และยังไม่มี MAE/RMSE

ขั้นต่อไป: สร้าง baseline และฝึกแต่ละโมเดลใน rolling-origin โดย filter target_date <= origin ก่อน fit เสมอ
ใช้ชุดประเมินร่วมกันสำหรับเปรียบเทียบหลัก และแสดงกลุ่มประวัติสั้นแยกต่างหาก
"""
(REPORT_DIR / "feature_summary.md").write_text(summary, encoding="utf-8")
display(Markdown(summary))
print("ข้อมูล:", MODEL_DATA_DIR)
print("สรุป:", REPORT_DIR)

# ผลเตรียมข้อมูลพยากรณ์ตลาดและอัตราการเข้าพัก

ใช้ weekly_market_cleaned.csv และ projects_cleaned.csv ซึ่งเป็นข้อมูลจำลอง

- ตารางรายสัปดาห์ 2,288 แถว 18 โครงการ
- ตาราง Random Forest 12 ชุด และ SARIMAX 12 ชุด สำหรับ 4 targets × 3 horizons
- Numeric features 33 ตัว และ categorical features 3 ตัว
- ราคา target ว่าง 3 แถว ไม่ถูกเติมเป็นคำตอบ
- origin แรกของ Validation ที่ h=12 คือ 2025-10-14; มี Train 165 แถวที่ยังไม่ทราบคำตอบและต้องกันออกจากรอบนี้
- ทดลองฤดูกาล 52 สัปดาห์ และเกณฑ์ชุดเปรียบเทียบหลัก 104 จุดที่มี y; เป็นค่าตั้งต้นของโครงงานที่ยังต้องประเมิน
- โมเดลที่ตกลงทดลอง: SARIMA, SARIMAX, Random Forest; baseline: Naïve, Seasonal Naïve
- ยังไม่ได้ fit imputer, encoder, scaler หรือโมเดล และยังไม่มี MAE/RMSE

ขั้นต่อไป: สร้าง baseline และฝึกแต่ละโมเดลใน rolling-origin โดย filter target_date <= origin ก่อน fit เสมอ
ใช้ชุดประเมินร่วมกันสำหรับเปรียบเทียบหลัก และแสดงกลุ่มประวัติสั้นแยกต่างหาก


ข้อมูล: D:\final project_\data\modeling\market_occupancy
สรุป: D:\final project_\reports\market_occupancy\features
